In [ ]:
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(dotenv_path=Path(".env"))
huggingface_token = os.environ.get("HUGGINGFACE_TOKEN")

if not huggingface_token:
    raise ValueError("HUGGINGFACE_TOKEN 이 .env 파일에 없습니다.")

login(token=huggingface_token)


In [ ]:
import torch
import wandb

from sklearn.model_selection import train_test_split

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    pipeline,
    Trainer,
)
from transformers.integrations import WandbCallback
from trl import DataCollatorForCompletionOnlyLM
import evaluate

# 모델과 토크나이저 불러오기
model_name = "google/gemma-2b-it"
use_cuda = torch.cuda.is_available()
use_bf16 = use_cuda and torch.cuda.is_bf16_supported()
model_dtype = torch.bfloat16 if use_bf16 else (torch.float16 if use_cuda else torch.float32)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=model_dtype,
)

if use_cuda:
    model = model.to("cuda")

tokenizer = AutoTokenizer.from_pretrained(model_name)


### GPU 사용 확인 메모

- `device_map="auto"`는 `accelerate`를 이용해 모델을 자동으로 적절한 장치에 배치할 때 주로 사용합니다.
- 현재 노트북은 GPU가 실제로 보이고 있는지와 모델이 어느 장치에 올라갔는지 확인하기 쉽도록 `torch.cuda.is_available()`와 `model.to("cuda")`를 사용합니다.
- 아래 셀은 모델 파라미터가 실제로 어느 장치에 올라가 있는지와, GPU 메모리가 사용되고 있는지를 빠르게 확인하기 위한 점검용 셀입니다.


In [ ]:
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
    print("current device:", torch.cuda.current_device())

print("model device:", next(model.parameters()).device)

if torch.cuda.is_available():
    print("allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
    print("reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))


In [ ]:
import datasets

dataset = datasets.load_dataset("jaehy12/news3")
element = dataset["train"][1]
element


In [ ]:
input_text = """다음 텍스트를 한국어로 간단히 요약해주세요:\n부산의 한 왕복 2차선 도로에서 역주행 사고로 배달 오토바이 운전자인 고등학생이 숨지는 사고가 발생했다.
유족은 '가해자가 사고 후 곧바로 신고하지 않고 늑장 대응해 피해를 키웠다'고 주장하고 있다.
11일 부산진경찰서는 교통사고처리특례법(교통사고처리법)상 업무상 과실치사 혐의로 지난 3일 A(59)씨를 검찰에 불구속 송치했다고 밝혔다.
A씨는 교통사고처리법상 12대 중과실에 해당되는 '중앙선 침범'으로 역주행 교통사고를 일으킨 혐의를 받는다.
경찰에 따르면 스포츠유틸리티차량(SUV) 운전자 A씨는 5월 19일 밤 11시 50분쯤 부산진구 가야고가교 밑 도로에서 중앙선을 넘어 역주행으로 140m를 달려
반대편 차선의 오토바이 운전자 조모(16)군을 들이받았다. 조군은 원동기장치자전거 면허를 취득한 상태였고 헬멧도 쓰고 있었지만 크게 다쳤다.
사고 당일 수술을 받았으나 얼마 후 2차 뇌출혈로 뇌사 판정이 내려졌고, 사고 발생 약 한 달 만인 지난달 16일 끝내 사망했다.
사고를 낸 A씨는 술을 마시거나 약물을 복용한 상태에서 운전하지는 않은 것으로 조사됐다.
경찰 관계자는 'A씨가 자신이 정주행을 하고 오토바이가 역주행을 한 것으로 착각했다고 진술했다'고 설명했다."""


def change_inference_chat_format(input_text):
    return [
        {"role": "user", "content": f"{input_text}"},
        {"role": "assistant", "content": """부산의 한 왕복 2차선 도로에서 역주행 사고로 배달 오토바이 운전자인 고등학생이 숨지는 사고가 발생했다.
     유족은 '가해자가 사고 후 곧바로 신고하지 않고 늑장 대응해 피해를 키웠다'고 주장하고 있다."""},
        {"role": "user", "content": "중요한 키워드 5개를 뽑아주세요."},
    ]


prompt = change_inference_chat_format(input_text)
# 채팅 템플릿을 문자열로 만든 뒤 다시 토크나이즈해서 generate 입력 형식에 맞춘다.
prompt_text = tokenizer.apply_chat_template(
    prompt,
    add_generation_prompt=True,
    tokenize=False
)
model_inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **model_inputs,
    max_new_tokens=256
)
generated_tokens = outputs[0][model_inputs["input_ids"].shape[-1]:]
print(tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
))


In [9]:
# Input_text : 위 정의된 기사와 동일

def change_inference_chat_format(input_text):
    return [
        {"role": "user", "content": f"{input_text}"},
        {"role": "assistant", "content": "한국어 요약:\n"}
    ]


# chat template 적용
prompt = change_inference_chat_format(input_text)

# 채팅 템플릿을 문자열로 만든 뒤 다시 토크나이즈해서 generate 입력 형식에 맞춘다.
prompt_text = tokenizer.apply_chat_template(
    prompt,
    add_generation_prompt=True,
    tokenize=False
)
model_inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

outputs = model.generate(**model_inputs, max_new_tokens=256, use_cache=True)
generated_tokens = outputs[0][model_inputs["input_ids"].shape[-1]:]
print(tokenizer.decode(generated_tokens, skip_special_tokens=True))


부산의 한 왕복 2차선 도로에서 역주행 사고로 배달 오토바이 운전자인 고등학생이 숨지는 사고가 발생했다. 그는 140m를 달려 반대편 차선의 오토바이 운전자에게 들려졌으나, 2차 뇌출혈로 뇌사 판정이 내려졌고 사망했다.
